# EqualCare — ML Backend Pipeline
**Input:** State + Disease

**Features used:** Hospital_Rating, District_Avg_Rating, Hospital_Score, Affordability_Score

**Output:** Top 5 Districts with Recommendation

⚠️ Run inside your venv (Python 3.11), NOT Python 3.14

Run each cell top to bottom ⬇️

## Step 1 — Install libraries (run once)

In [ ]:
pip install pandas scikit-learn numpy

## Step 2 — Import libraries

In [ ]:
import pandas as pd
import numpy as np
import pickle
import os

from sklearn.ensemble import GradientBoostingClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, classification_report

print('Libraries imported!')

## Step 3 — Load and clean data

In [ ]:
# Load CSV — update path if needed
df = pd.read_csv('equalcare_expanded_1.csv')

# Strip whitespace from text columns
for col in df.select_dtypes(include='object').columns:
    df[col] = df[col].str.strip()

# Remove duplicates
df.drop_duplicates(inplace=True)
df.reset_index(drop=True, inplace=True)

# Drop rows where Recommendation is missing (this was the NaN error in your notebook)
df.dropna(subset=['Recommendation'], inplace=True)
df.reset_index(drop=True, inplace=True)

print(f'Rows after cleaning : {len(df)}')
print(f'Recommendation values: {df["Recommendation"].unique()}')
df.head(3)

## Step 4 — Encode target and train model using the 4 key features

In [ ]:
# These are your 4 prediction features
FEATURES = ['Hospital_Rating', 'District_Avg_Rating', 'Hospital_Score', 'Affordability_Score']

X = df[FEATURES]

# Encode target: text labels → numbers
le = LabelEncoder()
y  = le.fit_transform(df['Recommendation'])

print('Classes:', list(le.classes_))
print('X shape:', X.shape)

# Train/test split — no stratify to avoid NaN issues
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Train GradientBoostingClassifier (matches your screenshot)
model = GradientBoostingClassifier(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

# Accuracy
y_pred = model.predict(X_test)
acc    = accuracy_score(y_test, y_pred)
print(f'\nModel Accuracy: {acc*100:.2f}%')
print()
print(classification_report(y_test, y_pred, target_names=le.classes_))

## Step 5 — Top 5 Districts function

In [ ]:
SCORE_MAP = {'Highly Recommended': 4, 'Recommended': 3, 'Moderate': 2, 'Not Recommended': 1}

def get_top5_districts(state, disease, dataframe, clf, encoder):
    """
    Input : state, disease, dataframe, trained model, label encoder
    Output: Top 5 districts as a DataFrame
    """
    # Filter by state
    f = dataframe[dataframe['State'] == state].copy()

    # Filter by disease
    f = f[f['Suitable_For_Disease'].str.contains(disease, case=False, na=False)]

    if f.empty:
        return None

    # ML predicts label for each hospital using the 4 features
    f['Pred_Label']  = encoder.inverse_transform(clf.predict(f[FEATURES]))
    f['Label_Score'] = f['Pred_Label'].map(SCORE_MAP)

    # Group by district, calculate averages
    grp = f.groupby('District').agg(
        Avg_Rating      = ('Hospital_Rating',     'mean'),
        Avg_Dist_Rating = ('District_Avg_Rating', 'mean'),
        Avg_Score       = ('Hospital_Score',      'mean'),
        Avg_Afford      = ('Affordability_Score', 'mean'),
        Avg_Label_Score = ('Label_Score',         'mean'),
        Total_Hospitals = ('Hospital_Rating',     'count')
    ).reset_index()

    # Combined ranking score
    grp['Combined_Score'] = (
        grp['Avg_Rating'] + grp['Avg_Dist_Rating'] +
        grp['Avg_Score']  + grp['Avg_Afford'] +
        grp['Avg_Label_Score'] * 2
    ) / 6

    # Best label per district
    best_label = f.groupby('District')['Pred_Label'].agg(
        lambda x: x.value_counts().idxmax()
    ).reset_index().rename(columns={'Pred_Label': 'Top_Label'})

    grp = grp.merge(best_label, on='District')
    top5 = grp.sort_values('Combined_Score', ascending=False).head(5).reset_index(drop=True)
    return top5.round(2)

print('Function ready!')

## Step 6 — Test the function

In [ ]:
# Change these to test different combinations
result = get_top5_districts('Karnataka', 'Cardiac Care', df, model, le)
print('Top 5 Districts in Karnataka for Cardiac Care:')
result

## Step 7 — Save everything with pickle
This creates the files your Streamlit app needs

In [ ]:
# Build states and diseases lists for frontend dropdowns
states = sorted(df['State'].unique().tolist())

all_diseases = set()
for v in df['Suitable_For_Disease']:
    for d in str(v).split(','):
        all_diseases.add(d.strip())
diseases = sorted(all_diseases)

# Save model
with open('model.pkl', 'wb') as f:
    pickle.dump(model, f)

# Save label encoder
with open('encoder.pkl', 'wb') as f:
    pickle.dump(le, f)

# Save cleaned dataframe
with open('hospital_data.pkl', 'wb') as f:
    pickle.dump(df, f)

# Save metadata for dropdowns and accuracy display
meta = {
    'states'  : states,
    'diseases': diseases,
    'accuracy': round(acc * 100, 2),
    'total_hospitals' : len(df),
    'total_states'    : df['State'].nunique(),
    'total_districts' : df['District'].nunique()
}
with open('meta.pkl', 'wb') as f:
    pickle.dump(meta, f)

print('model.pkl        saved ✅')
print('encoder.pkl      saved ✅')
print('hospital_data.pkl saved ✅')
print('meta.pkl         saved ✅')

## Step 8 — Verify all pickle files load correctly

In [ ]:
# This is exactly what app.py will do at startup — test it here first
m    = pickle.load(open('model.pkl',         'rb'))
enc  = pickle.load(open('encoder.pkl',       'rb'))
data = pickle.load(open('hospital_data.pkl', 'rb'))
mt   = pickle.load(open('meta.pkl',          'rb'))

print('Model type   :', type(m).__name__)
print('Classes      :', list(enc.classes_))
print('Data rows    :', len(data))
print('States       :', len(mt['states']))
print('Diseases     :', len(mt['diseases']))
print('Accuracy     :', mt['accuracy'], '%')
print('Total Hosp.  :', mt['total_hospitals'])
print('Total Dist.  :', mt['total_districts'])
print()
print('All pickle files verified! Ready to run app.py ✅')